In [ ]:
!wget https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz
!tar -xzf NLMCXR_reports.tgz
!find . -name "*.xml" | head -10

--2026-08-03 06:59:40--  https://openi.nlm.nih.gov/imgs/collections/NLMCXR_reports.tgz
Resolving openi.nlm.nih.gov (openi.nlm.nih.gov)... 130.14.65.157, 2607:f220:41e:7065::157
Connecting to openi.nlm.nih.gov (openi.nlm.nih.gov)|130.14.65.157|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1112632 (1.1M) [application/x-gzip]
Saving to: ‘NLMCXR_reports.tgz’

NLMCXR_reports.tgz  100%[===================>]   1.06M  4.51MB/s    in 0.2s    

2026-08-03 06:59:44 (4.51 MB/s) - ‘NLMCXR_reports.tgz’ saved [1112632/1112632]

./ecgen-radiology/2333.xml
./ecgen-radiology/2043.xml
./ecgen-radiology/3535.xml
./ecgen-radiology/3646.xml
./ecgen-radiology/1102.xml
./ecgen-radiology/3888.xml
./ecgen-radiology/2613.xml
./ecgen-radiology/1262.xml
./ecgen-radiology/356.xml
./ecgen-radiology/3326.xml


In [ ]:
import os
import xml.etree.ElementTree as ET

def clean_text(text):
    if text is None:
        return ""
    text = text.replace("XXXX", "")  # some reports have placeholder text
    text = " ".join(text.split())     # remove extra spaces and newlines
    return text


"""
    open one XML file, extract FINDINGS and IMPRESSION.
    returns a dictionary with reportID, findings, impression, and combined text.
    This becomes my data, can loop through it, feed each report into the model and run the cheXpert labeler.
    """
def extract_report(xml_path):
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()

        report_id = xml_path.split("/")[-1].replace(".xml", "")

        findings = ""
        impression = ""


        for abstract in root.findall(".//AbstractText"):
            label = abstract.get("Label")
            text = abstract.text if abstract.text else ""  # gets the text inside the tag

            if label == "FINDINGS":
                findings = clean_text(text)
            elif label == "IMPRESSION":
                impression = clean_text(text)

        return {
            "id": report_id,
            "findings": findings,
            "impression": impression,
            "full_report": f"FINDINGS: {findings}\nIMPRESSION: {impression}"
        }

    except Exception as e:
        print(f"Error processing {xml_path}: {e}")
        return None

# folder where extracted XML files are
xml_folder = "./ecgen-radiology/"

# data
reports = []

# loop through every file in the folder
for filename in os.listdir(xml_folder):
    if filename.endswith(".xml"):
        path = os.path.join(xml_folder, filename)
        report = extract_report(path)
        if report:
            reports.append(report)


print(f"Processed {len(reports)} reports")

import pandas as pd
df = pd.DataFrame(reports)
df.to_csv("processed_reports.csv", index=False)

Processed 3955 reports


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df.to_csv("/content/drive/MyDrive/processed_reports.csv", index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

df.head()

,id,findings,impression,full_report
0,2333,Heart size is normal. No focal consolidations....,No acute cardiopulmonary findings.,FINDINGS: Heart size is normal. No focal conso...
1,2043,,Considering differences in technical factors s...,FINDINGS: \nIMPRESSION: Considering difference...
2,3535,The lungs are clear. The heart and pulmonary a...,No acute cardiopulmonary disease,FINDINGS: The lungs are clear. The heart and p...
3,3646,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...,FINDINGS: The lungs are clear. Heart size is n...
4,1102,There is stable cardiomegaly with pulmonary va...,"1. Cardiomegaly, vascular congestion and proba...",FINDINGS: There is stable cardiomegaly with pu...


In [ ]:
# Load the CheXpert Labeler to generate the ground truth labels

!git clone https://github.com/stanfordmlgroup/chexpert-labeler.git
%cd chexpert-labeler

fatal: destination path 'chexpert-labeler' already exists and is not an empty directory.
/content/chexpert-labeler


In [ ]:
!cat environment.yml

name: chexpert-label
channels:
  - anaconda
  - conda-forge
  - defaults
dependencies:
  - blas=1.0=mkl
  - ca-certificates=2020.1.1=0
  - certifi=2019.11.28=py36_0
  - decorator=4.3.0=py_0
  - docopt=0.6.2=py_1
  - gcc_linux-64=7.3.0=h553295d_9
  - gxx_linux-64=7.3.0=h553295d_9
  - intel-openmp=2019.1=144
  - jpype1=0.6.3=py36h9de70de_1001
  - libblas=3.8.0=14_mkl
  - libcblas=3.8.0=14_mkl
  - libedit=3.1.20170329=hf8c457e_1001
  - libffi=3.2.1=he1b5a44_1006
  - libgcc-ng=8.2.0=hdf63c60_1
  - libgfortran-ng=7.3.0=hdf63c60_5
  - liblapack=3.8.0=14_mkl
  - libstdcxx-ng=8.2.0=hdf63c60_1
  - mkl=2019.1=144
  - mkl_fft=1.0.6=py36_0
  - mkl_random=1.0.1=py36_0
  - ncurses=6.1=hf484d3e_1002
  - networkx=1.11=py36_0
  - nltk=3.3.0=py36_0
  - numpy=1.15.4=py36h8b7e671_1002
  - numpy-base=1.15.4=py36hde5b4d6_0
  - openssl=1.1.1f=h7b6447c_0
  - pandas=0.23.4=py36h637b7d7_1000
  - pathlib2=2.3.5=py36_0
  - pip=18.1=py36_1000
  - ply=3.11=py_1
  - python=3.6.7=h0371630_0
  - python-dateutil=2.7.5=

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [ ]:
!conda env create -f environment.yml

Retrieving notices: - \ done
Channels:
 - anaconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: \ | done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.0
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda



mkl-2019.1           | 204.6 MB  | :   0% 0/1 [00:00<?, ?it/s]
gcc_impl_linux-64-7. | 73.2 MB   | :   0% 0/1 [00:00<?, ?it/s]

python-3.6.7         | 30.1 MB   | :   0% 0/1 [00:00<?, ?it/s]


gxx_impl_linux-64-7. | 18.7 MB   | :   0% 0/1 [00:00<?, ?it/s]



binutils_impl_linux- | 16.5 MB   | :   0% 0/1 [00:00<?, ?it/s]




pandas-0.23.4        | 9.8 MB    | :   0% 0/1 [00:00<?, ?it/s]





libgcc-ng-8.2.0      | 7.6 MB    | :   0% 0/1 [00:00<?, ?it/s]






numpy-base-1.15.4    | 4.3 MB    | :   0% 0/1 [00:00<?, ?it/s]







numpy-1.15.4         | 4.2 MB    | :   0% 0/1 [00:00<?, ?it/s]








openssl-1.1.1f       | 3.8 MB    | :   0% 0/1 [00:00<?, ?it/s]









In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/processed_reports.csv")

df[['full_report']].to_csv("chexpert_input.csv", index=False, header=False)

In [ ]:
!conda run -n chexpert-label pip install --no-deps git+https://github.com/ncbi-nlp/negbio.git

  Cloning https://github.com/ncbi-nlp/negbio.git to /tmp/pip-req-build-k0z54996
  Running setup.py bdist_wheel for negbio: started
  Running setup.py bdist_wheel for negbio: finished with status 'done'
  Stored in directory: /tmp/pip-ephem-wheel-cache-obds5dqi/wheels/ac/3f/30/0103a413ffdcb47844e7806081dd46b363b30844dacd4c555d
Successfully built negbio



In [ ]:
!conda run -n chexpert-label python -m bllipparser.ModelFetcher -i GENIA+PubMed

Model directory: /root/.local/share/bllipparser/GENIA+PubMed
Fetching model: GENIA+PubMed from https://www.dropbox.com/s/ev3h78gq7526xdj/BLLIP-GENIA-PubMed.tar.bz2?dl=1
Downloaded 0.0% (0.0 MB)
Downloaded 0.0% (0.0 MB)
Downloaded 0.1% (0.0 MB)
Downloaded 0.1% (0.0 MB)
Downloaded 0.1% (0.0 MB)
Downloaded 0.1% (0.0 MB)
Downloaded 0.2% (0.0 MB)
Downloaded 0.2% (0.1 MB)
Downloaded 0.2% (0.1 MB)
Downloaded 0.3% (0.1 MB)
Downloaded 0.3% (0.1 MB)
Downloaded 0.3% (0.1 MB)
Downloaded 0.3% (0.1 MB)
Downloaded 0.4% (0.1 MB)
Downloaded 0.4% (0.1 MB)
Downloaded 0.4% (0.1 MB)
Downloaded 0.5% (0.1 MB)
Downloaded 0.5% (0.1 MB)
Downloaded 0.5% (0.1 MB)
Downloaded 0.6% (0.1 MB)
Downloaded 0.6% (0.2 MB)
Downloaded 0.6% (0.2 MB)
Downloaded 0.6% (0.2 MB)
Downloaded 0.7% (0.2 MB)
Downloaded 0.7% (0.2 MB)
Downloaded 0.7% (0.2 MB)
Downloaded 0.8% (0.2 MB)
Downloaded 0.8% (0.2 MB)
Downloaded 0.8% (0.2 MB)
Downloaded 0.8% (0.2 MB)
Downloaded 0.9% (0.2 MB)
Downloaded 0.9% (0.2 MB)
Downloaded 0.9% (0.2 MB)
Downlo

In [ ]:
!nohup conda run -n chexpert-label python -u label.py --reports_path chexpert_input.csv --output_path chexpert_labels.csv --verbose > run_log.txt 2>&1 &

In [ ]:
import pandas as pd
df_labels = pd.read_csv("chexpert_labels.csv")
print(df_labels.shape)
df_labels.head()

(3955, 15)


,Reports,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Lesion,Lung Opacity,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,FINDINGS: Heart size is normal. No focal conso...,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN
1,FINDINGS: \nIMPRESSION: Considering difference...,NaN,-1.0,1.0,1.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,FINDINGS: The lungs are clear. The heart and p...,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FINDINGS: The lungs are clear. Heart size is n...,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4,FINDINGS: There is stable cardiomegaly with pu...,NaN,NaN,1.0,NaN,1.0,-1.0,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN


In [ ]:
# Merge reports with labes
import pandas as pd

df_original = pd.read_csv("/content/drive/MyDrive/processed_reports.csv")
df_labels = pd.read_csv("chexpert_labels.csv")

assert len(df_original) == len(df_labels), f"Mismatch: {len(df_original)} vs {len(df_labels)}"

df_original = df_original.reset_index(drop=True)
df_labels = df_labels.reset_index(drop=True)

label_cols = df_labels.drop(columns=['Reports'])

df_final = pd.concat([df_original, label_cols], axis=1)

print(df_final.shape)
df_final.head()

(3955, 18)


,id,findings,impression,full_report,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Lesion,Lung Opacity,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,2333,Heart size is normal. No focal consolidations....,No acute cardiopulmonary findings.,FINDINGS: Heart size is normal. No focal conso...,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,0.0,0.0,NaN,NaN,NaN
1,2043,NaN,Considering differences in technical factors s...,FINDINGS: \nIMPRESSION: Considering difference...,NaN,-1.0,1.0,1.0,NaN,0.0,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,3535,The lungs are clear. The heart and pulmonary a...,No acute cardiopulmonary disease,FINDINGS: The lungs are clear. The heart and p...,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3646,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...,FINDINGS: The lungs are clear. Heart size is n...,1.0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4,1102,There is stable cardiomegaly with pulmonary va...,"1. Cardiomegaly, vascular congestion and proba...",FINDINGS: There is stable cardiomegaly with pu...,NaN,NaN,1.0,NaN,1.0,-1.0,NaN,NaN,NaN,0.0,1.0,NaN,NaN,NaN


In [ ]:
# to confirm alignment held
for i in [0, 1, 100, 2000, 3954]:
    match = df_original['full_report'].iloc[i].strip() == df_labels['Reports'].iloc[i].strip()
    print(f"Row {i}: match = {match}")

Row 0: match = True
Row 1: match = True
Row 100: match = True
Row 2000: match = True
Row 3954: match = True


In [ ]:
df_final.to_csv("/content/drive/MyDrive/final_labeled_reports.csv", index=False)